<a href="https://colab.research.google.com/github/BlancaHoma14/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion10/sesion10_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 10 · Hito 1: Mini-Evaluador de Expresiones y AST Completo

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Puede un programa leer una expresión escrita y devolver su valor sin que nadie la evalúe antes?
>
> **Objetivo de la sesión:** Integrar lo aprendido en la unidad en un mini-evaluador que lee una expresión desde texto, respeta precedencia y paréntesis, y reporta errores de sintaxis y división entre cero.

**Entregable:** tu evaluador completo en artesano_sesion10_c.c, el harness en verde y la bitácora del hito

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

El analizador ya viene armado: solo completas primaria, termino y expr. Debes leer enteros, + - * /, paréntesis y espacios. error=1 es división entre cero; error=2, sintaxis inválida.

Al final de la sesión tendrás: **Un mini-evaluador que convierte 2+3*4 en 14, respeta paréntesis y distingue error de sintaxis de división entre cero.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 10](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion10.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion10.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion10.h"

long evaluar_expresion(const char *fuente, int *error);
```

- Sin variables ni menos unario: solo enteros y los operadores + - * /.
- Respeta precedencia y paréntesis.
- Completa las tres funciones marcadas; no cambies la firma de evaluar_expresion.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion10_c.c` junto con `artesano_sesion10_c.c`
> (o `ia_sesion10_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `"2+3*4"` | 14 |
| 2 | `"(2+3)*4"` | 20 |
| 3 | `"10/0" -> error 1` | 1 |
| 4 | `"2+" -> error 2` | 2 |
| 5 | `"  7 "` | 7 |
| 6 | `"5-10"` | -5 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El analizador respeta precedencia: la multiplicación se resuelve antes que la suma.
- [ ] Los paréntesis y los espacios no alteran el resultado.
- [ ] La división entre cero y la sintaxis inválida se reportan con códigos distintos.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion10_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion10.h
/* api_sesion10.h - contrato de la Sesion 10
 * Mini-evaluador de expresiones (Hito 1)
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION10_H
#define API_SESION10_H

long evaluar_expresion(const char *fuente, int *error);

#endif /* API_SESION10_H */


Writing api_sesion10.h


In [3]:
%%writefile artesano_sesion10_c.c
/* artesano_sesion10_c.c - Fase 1: El Artesano
 * Mini-evaluador de expresiones (Hito 1).
 */

#include "api_sesion10.h"
#include <stdio.h>
#include <string.h>
#include <ctype.h>

/* Estructura del analizador */
typedef struct {
    const char *s;
    int error;
} Parser;

/* Ignorar espacios */
static void saltar(Parser *p) {
    while (*p->s == ' ') {
        p->s++;
    }
}

long expr(Parser *p);

/* Leer numeros y resolver parentesis */
long primaria(Parser *p) {
    long valor = 0;

    saltar(p);

    /* Resolver parentesis */
    if (*p->s == '(') {
        p->s++;
        valor = expr(p);
        saltar(p);

        if (*p->s == ')') {
            p->s++;
        } else {
            p->error = 2;
        }

        return valor;
    }

    /* Leer un numero entero */
    if (isdigit((unsigned char)*p->s)) {
        while (isdigit((unsigned char)*p->s)) {
            valor = valor * 10 + (*p->s - '0');
            p->s++;
        }

        return valor;
    }

    /* Sintaxis invalida */
    p->error = 2;
    return 0;
}

/* Resolver multiplicaciones y divisiones */
long termino(Parser *p) {
    long valor = primaria(p);

    while (p->error == 0) {
        long siguiente;

        saltar(p);

        if (*p->s == '*') {
            p->s++;
            siguiente = primaria(p);
            valor = valor * siguiente;
        } else if (*p->s == '/') {
            p->s++;
            siguiente = primaria(p);

            if (p->error != 0) {
                break;
            }

            if (siguiente == 0) {
                p->error = 1;
                break;
            }

            valor = valor / siguiente;
        } else {
            break;
        }
    }

    return valor;
}

/* Resolver sumas y restas */
long expr(Parser *p) {
    long valor = termino(p);

    while (p->error == 0) {
        long siguiente;

        saltar(p);

        if (*p->s == '+') {
            p->s++;
            siguiente = termino(p);
            valor = valor + siguiente;
        } else if (*p->s == '-') {
            p->s++;
            siguiente = termino(p);
            valor = valor - siguiente;
        } else {
            break;
        }
    }

    return valor;
}

/* Funcion principal del evaluador */
long evaluar_expresion(const char *fuente, int *error) {
    static int sin_error;

    if (error == NULL) {
        error = &sin_error;
    }

    *error = 0;

    if (fuente == NULL) {
        *error = 2;
        return 0;
    }

    Parser p = { fuente, 0 };

    long valor = expr(&p);

    saltar(&p);

    if (p.error == 0 && *p.s != '\0') {
        p.error = 2;
    }

    *error = p.error;

    return p.error != 0 ? 0 : valor;
}

Writing artesano_sesion10_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion10_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


**CONTRATO C-R-E-O**
**CONTEXTO:** Estoy trabajando en la Sesión 10 de Lenguajes y Autómatas II.
Necesito desarrollar un mini-evaluador de expresiones aritméticas
en lenguaje C, utilizando un analizador recursivo descendente.

**ROL:** Actúa como un ingeniero de software experto en lenguaje C,
compiladores, análisis sintáctico y evaluación de expresiones.

**ESPECIFICACIÓN:** Implementa y completa las funciones primaria(), termino()
y expr() utilizando la estructura Parser proporcionada.

El programa debe:
1. Leer números enteros desde una cadena de texto.
2. Reconocer los operadores +, -, * y /.
3. Respetar la precedencia de operadores.
4. Resolver expresiones dentro de paréntesis.
5. Ignorar los espacios entre los elementos.
6. Reportar error 1 cuando exista división entre cero.
7. Reportar error 2 cuando exista sintaxis inválida.
8. Propagar los errores sin continuar realizando operaciones.
9. Detectar caracteres sobrantes al terminar de analizar.
10. Mantener la firma:
    long evaluar_expresion(const char *fuente, int *error);
11. No incluir main(), porque el harness ya lo proporciona.
12. No modificar el contrato ni el harness.
13. Compilar con:
    gcc -std=c11 -Wall -Wextra -Werror

**OUTPUT:** Entrega el código completo para ia_sesion10_c.c.
Explica las mejoras respecto a una implementación manual,
los casos límite considerados y la complejidad temporal
y espacial.

In [4]:
%%writefile ia_sesion10_c.c
/* ia_sesion10_c.c - Fase 2: El Cientifico
 * Mini-evaluador de expresiones (Hito 1).
 */

#include "api_sesion10.h"
#include <stdio.h>
#include <string.h>
#include <ctype.h>

typedef struct {
    const char *s;
    int error;
} Parser;

/* Ignorar espacios */
static void saltar(Parser *p) {
    while (*p->s == ' ') {
        p->s++;
    }
}

long expr(Parser *p);

/* Leer numeros y resolver parentesis */
long primaria(Parser *p) {
    long valor = 0;

    saltar(p);

    /* Resolver expresiones entre parentesis */
    if (*p->s == '(') {
        p->s++;

        valor = expr(p);

        /* No continuar si ya existe un error */
        if (p->error != 0) {
            return 0;
        }

        saltar(p);

        if (*p->s != ')') {
            p->error = 2;
            return 0;
        }

        p->s++;
        return valor;
    }

    /* Leer un numero entero */
    if (isdigit((unsigned char)*p->s)) {
        while (isdigit((unsigned char)*p->s)) {
            valor = valor * 10 + (*p->s - '0');
            p->s++;
        }

        return valor;
    }

    /* Reportar error de sintaxis */
    p->error = 2;
    return 0;
}

/* Resolver multiplicaciones y divisiones */
long termino(Parser *p) {
    long valor = primaria(p);

    while (p->error == 0) {
        char operador;
        long siguiente;

        saltar(p);
        operador = *p->s;

        if (operador != '*' && operador != '/') {
            break;
        }

        p->s++;
        siguiente = primaria(p);

        /* Propagar errores */
        if (p->error != 0) {
            return 0;
        }

        if (operador == '*') {
            valor *= siguiente;
        } else {
            if (siguiente == 0) {
                p->error = 1;
                return 0;
            }

            valor /= siguiente;
        }
    }

    return valor;
}

/* Resolver sumas y restas */
long expr(Parser *p) {
    long valor = termino(p);

    while (p->error == 0) {
        char operador;
        long siguiente;

        saltar(p);
        operador = *p->s;

        if (operador != '+' && operador != '-') {
            break;
        }

        p->s++;
        siguiente = termino(p);

        /* No operar si el analizador detecto un error */
        if (p->error != 0) {
            return 0;
        }

        if (operador == '+') {
            valor += siguiente;
        } else {
            valor -= siguiente;
        }
    }

    return valor;
}

/* Funcion principal del evaluador */
long evaluar_expresion(const char *fuente, int *error) {
    static int sin_error;

    if (error == NULL) {
        error = &sin_error;
    }

    *error = 0;

    if (fuente == NULL) {
        *error = 2;
        return 0;
    }

    Parser p = { fuente, 0 };

    long valor = expr(&p);

    saltar(&p);

    /* Detectar caracteres sobrantes */
    if (p.error == 0 && *p.s != '\0') {
        p.error = 2;
    }

    *error = p.error;

    return p.error != 0 ? 0 : valor;
}


Writing ia_sesion10_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion10_c.c
/* harness_sesion10_c.c - auditoria automatica de la Sesion 10
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c artesano_sesion10_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion10.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int e=0; long r=evaluar_expresion("2+3*4",&e); (void)e; verificar_int("\"2+3*4\"", (long)(14), (long)(r)); }
    { int e=0; long r=evaluar_expresion("(2+3)*4",&e); (void)e; verificar_int("\"(2+3)*4\"", (long)(20), (long)(r)); }
    { int e=0; long r=evaluar_expresion("10/0",&e); (void)r; verificar_int("\"10/0\" -> error 1", (long)(1), (long)(e)); }
    { int e=0; long r=evaluar_expresion("2+",&e); (void)r; verificar_int("\"2+\" -> error 2", (long)(2), (long)(e)); }
    { int e=0; long r=evaluar_expresion("  7 ",&e); (void)e; verificar_int("\"  7 \"", (long)(7), (long)(r)); }
    { int e=0; long r=evaluar_expresion("5-10",&e); (void)e; verificar_int("\"5-10\"", (long)(-5), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md" "\n");
    return 0;
}


Writing harness_sesion10_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c artesano_sesion10_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c ia_sesion10_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion10.md` a `auditorias/auditoria_sesion10.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
